# Notebook 19 — Phase-Lock Survival Boundary

This notebook adds a final boundary-style diagnostic for the noisy phase-locked CZ solution.

Instead of looking at fidelity, leakage, coherence, and conditional phase separately, it asks a sharper question:

**Where does the phase-locked CZ gate still meaningfully survive under noise?**

We define a survival region using thresholds on:
- compensated CZ fidelity,
- coherence proxy,
- leakage proxy.

From those, we build:
1. a binary survival map,
2. a survival score map,
3. a boundary contour,
4. 1D survival slices.

This makes the final result easier to summarize:
- where the gate remains usable,
- where coherence still supports phase interpretation,
- where noise pushes the protocol outside the phase-locked regime.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Phase-locked protocol from Notebook 17d / 18b

In [ ]:
opt = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

for k, v in opt.items():
    print(f"{k}: {v:.6f}")


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, T, omega_max, alpha, V, delta0, p, q,
                        gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    H = build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q)
    times = np.linspace(0.0, T, n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            amp = basis_state.overlap(state)
            vals.append(amp)
        else:
            val = (basis_state.dag() * state * basis_state)
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(T, omega_max, alpha, V, delta0, p, q,
                              gamma_decay=0.0, gamma_phi=0.0, n_steps=220):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, T, omega_max, alpha, V, delta0, p, q,
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            amp = basis_state.overlap(final_state)
            score = np.abs(amp) ** 2
        else:
            val = (basis_state.dag() * final_state * basis_state)
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Thresholds for survival

In [ ]:
# You can adjust these thresholds later.
fidelity_threshold = 0.20
coherence_threshold = 0.005
leakage_threshold = 0.35

print("Survival thresholds:")
print(f"compensated CZ fidelity >= {fidelity_threshold:.3f}")
print(f"coherence proxy >= {coherence_threshold:.3f}")
print(f"leakage proxy <= {leakage_threshold:.3f}")


## 2D noise scan

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 25)
gamma_phi_vals = np.linspace(0.0, 0.12, 25)

cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

for i, gamma_phi in enumerate(gamma_phi_vals):
    for j, gamma_decay in enumerate(gamma_decay_vals):
        U_eff, finals = build_noisy_effective_map(
            opt['T'], opt['omega_max'], opt['alpha'], V, opt['delta0'], opt['p'], opt['q'],
            gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=200
        )
        cz_map[i, j] = compensated_cz_fidelity(U_eff)
        coh_map[i, j] = coherence_proxy(finals)
        leak_map[i, j] = leakage_from_finals(finals)


## Survival logic

In [ ]:
survival_mask = (
    (cz_map >= fidelity_threshold) &
    (coh_map >= coherence_threshold) &
    (leak_map <= leakage_threshold)
).astype(float)

# Smooth scalar score showing how comfortably a point survives.
survival_score = np.minimum.reduce([
    cz_map / fidelity_threshold,
    coh_map / coherence_threshold,
    leakage_threshold / np.maximum(leak_map, 1e-12),
])

# Boundary level = 1 means exactly on threshold.
boundary_level = 1.0

print("Fraction of noise grid surviving:", survival_mask.mean())


## Plot binary survival region

In [ ]:
plt.figure(figsize=(7.2, 5.4))
im = plt.imshow(
    survival_mask,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
    vmin=0, vmax=1
)
plt.contour(gamma_decay_vals, gamma_phi_vals, survival_mask, levels=[0.5], colors='white', linewidths=1.2)
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('Pure dephasing gamma_phi')
plt.title('Phase-lock survival region')
plt.colorbar(im, label='survival mask')
plt.tight_layout()
plt.show()


## Plot survival score and boundary

In [ ]:
plt.figure(figsize=(7.2, 5.4))
im = plt.imshow(
    survival_score,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
)
plt.contour(gamma_decay_vals, gamma_phi_vals, survival_score, levels=[boundary_level], colors='white', linewidths=1.2)
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('Pure dephasing gamma_phi')
plt.title('Phase-lock survival score')
plt.colorbar(im, label='survival score')
plt.tight_layout()
plt.show()


## Overlay boundary on core diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

im0 = axes[0].imshow(
    cz_map, origin='lower', aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[0].contour(gamma_decay_vals, gamma_phi_vals, survival_score, levels=[boundary_level], colors='white', linewidths=1.0)
axes[0].set_title('Compensated CZ fidelity')
axes[0].set_xlabel('gamma')
axes[0].set_ylabel('gamma_phi')

im1 = axes[1].imshow(
    coh_map, origin='lower', aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[1].contour(gamma_decay_vals, gamma_phi_vals, survival_score, levels=[boundary_level], colors='white', linewidths=1.0)
axes[1].set_title('Coherence proxy')
axes[1].set_xlabel('gamma')
axes[1].set_ylabel('gamma_phi')

im2 = axes[2].imshow(
    leak_map, origin='lower', aspect='auto',
    extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]]
)
axes[2].contour(gamma_decay_vals, gamma_phi_vals, survival_score, levels=[boundary_level], colors='white', linewidths=1.0)
axes[2].set_title('Leakage proxy')
axes[2].set_xlabel('gamma')
axes[2].set_ylabel('gamma_phi')

fig.colorbar(im2, ax=axes.ravel().tolist(), label='value')
plt.tight_layout()
plt.show()


## 1D survival slices

In [ ]:
slice_phi0_survive = survival_mask[0, :]
slice_decay0_survive = survival_mask[:, 0]

slice_phi0_score = survival_score[0, :]
slice_decay0_score = survival_score[:, 0]


In [ ]:
plt.figure(figsize=(7, 4.4))
plt.plot(gamma_decay_vals, slice_phi0_score, label='survival score')
plt.axhline(1.0, linestyle='--', label='boundary')
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('Survival score')
plt.title('Survival slice at gamma_phi = 0')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4.4))
plt.plot(gamma_phi_vals, slice_decay0_score, label='survival score')
plt.axhline(1.0, linestyle='--', label='boundary')
plt.xlabel('Pure dephasing gamma_phi')
plt.ylabel('Survival score')
plt.title('Survival slice at gamma = 0')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4.4))
plt.step(gamma_decay_vals, slice_phi0_survive, where='mid')
plt.xlabel('Spontaneous emission gamma')
plt.ylabel('Survival mask')
plt.title('Binary survival slice at gamma_phi = 0')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4.4))
plt.step(gamma_phi_vals, slice_decay0_survive, where='mid')
plt.xlabel('Pure dephasing gamma_phi')
plt.ylabel('Survival mask')
plt.title('Binary survival slice at gamma = 0')
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
survival_fraction = float(survival_mask.mean())

# approximate boundary intercepts on axes
gamma_axis_survive = gamma_decay_vals[slice_phi0_survive > 0.5]
gamma_phi_axis_survive = gamma_phi_vals[slice_decay0_survive > 0.5]

gamma_max_survive = float(gamma_axis_survive.max()) if gamma_axis_survive.size else 0.0
gamma_phi_max_survive = float(gamma_phi_axis_survive.max()) if gamma_phi_axis_survive.size else 0.0

summary_text = f'''
Phase-lock survival boundary summary

Protocol:
T      = {opt["T"]:.4f}
alpha  = {opt["alpha"]:.4f}
Omega  = {opt["omega_max"]:.4f}
Delta0 = {opt["delta0"]:.4f}
p      = {opt["p"]:.4f}
q      = {opt["q"]:.4f}

Thresholds:
compensated CZ fidelity >= {fidelity_threshold:.3f}
coherence proxy >= {coherence_threshold:.3f}
leakage proxy <= {leakage_threshold:.3f}

Grid summary:
survival fraction of noise plane = {survival_fraction:.4f}
max gamma surviving at gamma_phi = 0      -> {gamma_max_survive:.4f}
max gamma_phi surviving at gamma = 0      -> {gamma_phi_max_survive:.4f}

Interpretation:
- survival is concentrated near the low-noise corner,
- spontaneous emission limits survival more strongly than dephasing in this model,
- the phase-locked CZ regime persists only where fidelity, coherence, and leakage all remain jointly acceptable.
'''
print(summary_text)


## Final conclusion

This notebook turns the noisy phase-locked CZ analysis into a boundary question:

**where does the gate still survive as a usable phase-locked CZ solution?**

By combining compensated CZ fidelity, coherence retention, and leakage control into one survival condition, it produces a compact final figure:

- a surviving region,
- a survival score,
- and a boundary contour in the noise plane.

That gives a sharp final statement for the repo:

**the phase-locked shaped CZ solution exists as a coherent, low-leakage gate only in a bounded low-noise region, with spontaneous emission setting the tighter survival limit in this simple Lindblad model.**
